In [ ]:
!pip install pandas unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.2 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import unidecode
import functools

In [1]:
# Path
path = '/thibault_proprio/multq/rawdata/'

# Équivalent de la table 'B05V_ADR_PROPRIO'
csv = 'proprio_adresse.csv'

In [ ]:
# Renommer les colonnes
col_dict = {
    'anrole': 'anrole',
    'id_provinc': 'id_provinc',
    'code_mun': 'code_mun',
    'rl0102a': 'code_arr',
    'mat18': 'mat18',
    'rl0201ax': 'nom',
    'rl0201bx': 'prenom',
    'rl0201u': 'code_type',
    'no_coprop': 'no_coprop',
    'rl0201cx': 'adresse_post',
    'rl0201dx': 'muni',
    'rl0201ex': 'code_post',
    'rl0201fx': 'adresse_comp',
    'rl0201gx': 'date_insc',
    'rl0201hx': 'statut',
    'rl0201ix': 'num_civ',
    'rl0201jx': 'fraction',
    'rl0201kx': 'cpde_gen',
    'rl0201lx': 'code_lien',
    'rl0201mx': 'rue',
    'rl0201nx': 'orient',
    'rl0201ox': 'num_app',
    'rl0201px': 'fraction_num_app',
    'rl0201qx': 'prov',
    'rl0201rx': 'pays',
    'rl0201sx': 'succ',
    'rl0201tx': 'cp',
}

## Renommer les types de voie
replace_voie = {'AV' : "av",
                'RU' : "rue",
                'PL' : "boul",
                'CH' : "chemin",
                'BO' : "montee",
                'RO' : "route",
                'RA' : "rang",
                'TE' : "terrasse",
                'CT' : "cote",
                'PR' : "promenade",
                'CR' : "croissant",
                'IM' : "impasse",
                'CA' : "carré",
                'AL' : "allee",
                'LC' : "lac",
                'KR' : "cres",
                'SN' : "sentier",
                'AT' : "aut",
                'CI' : "cir",
                'DR' : "drive",
                'PU' : "plateau",
                'RL' : "ruelle",
                'CO' : "cours",
                'DE' : "desserte",
                'ST' : "street",
                'RD' : "road",
                'CE' : "cercle",
                'DO' : "domaine",
                'EP' : "esplanade",
                'KO' : "crt",
                'RG' : "ridg",
                'TV' : "transverse",
                'TA' : "terrasse",
                'GA' : "gdns",
                'CS' : "concessession",
                'LN' : "lane",
                'TC' : " trait-carre",
                'DS' : "descente",
                'AR' : "ancienne-route",
                'VO' : "voie",
                'JA' : "ja",}

In [ ]:
# Stopwords

road_type_fr = ["rue", "ch", "rte", "av", "boul", "road", "crois", "rd", "rang", "place", "boulevard", "terr", "drive",
                "cours", "montee", "montée", "terrasse", "imp", "prom", "promenade", "drive", "dr", 'al', 'allee', 'av.',
                'avenue', 'bl', 'boul.', 'carre', 'carré', 'chemin', 'cir', 'cour', 'court', 'cr', 'cro', 'ct', 'mte', 'mtee',
                'pl', 'pr', 'prom', 'rle', 'ruelle', 'tsse', "ch.", 'autoroute', 'aut.', "route", "lane", "way", "concession",
                "street"]

road_type_en = ["road", "rd", "drive", "dr", "lane", "way", "street", "cr"]

stopwords = [",", ".", "de", "du", "de la", "la", "l'", "des", "le"]

directions = ["ouest", "est", "nord", "sud", "south", "north", "east", "west", "sw", "nw"]

suite_stopwords = ['cp', 'c.p.', 'case postale', 'po box', 'p.o. box', 'p.o.box', 'bureau', 'suite', 'unit', 'appartement', 'app', 'app.', 'étage', 'etage', 'et.', 'ét.']

In [ ]:
# Expressions regex

## Extrait le numéro de rue
reg_num = r'(^(?=.*[0-9])(?=.*[a-zA-Z]))'
reg_digits = r'\d+'
reg_digits_hyphen = r'(\d+)-(\d+)'

## Extrait le casier postal
reg_cp = r"^([(cp|c.p.|case postale)\s]+ [\w\s]+)|^((cp|c.p.|case postale)(\w+))|^([(po box|p.o. box|p.o.box)\s]+ [\w\s]+)"

## Extrait le numéro de suite/de logement
reg_suite = r'(bureau|suite|cp|c.p.|unit|case postale|appartement|app|app.|po box|p.o. box|p.o.box|pmb)((\s||-)[a-zA-Z0-9_.-]*|\s(\w+)|-(\w+)|(\w+))'
reg_enieme = r'\d+(e|er|em|eme|nd|st|rst)(\s|-)(étage|ét-|ét|ét.|etage|floor)'


## Extrait le nom de la voie
reg_st = r'^(\s*)(saint)'
reg_ste = r'^(\s*)(sainte)'
reg_avenue = r"(\d+\s*(?:e|eme|er|th|ieme|iere|ere|re)\s(?:rang|avenue|rue|concession|chemin))|(?:route\s\d+)"
reg_route = r'(route|autoroute)\s*\d+'


In [ ]:
# Fonctions de nettoyage

def clean_text(x):
    if pd.isna(x):
        return x
    x = unidecode.unidecode(str(x)).lower()
    x = re.sub(r"[^A-Z0-9 ]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def has_numbers(input_string):
    return any(char.isdigit() for char in input_string)

reg_zip = r"(?:[a-z][0-9]){3}|[a-z]{1}[0-9]{1}[a-z]{1}\s*[0-9]{1}[a-z]{1}[0-9]{1}"

def clean_zip(data):
    zipcode = None
    zip_match = re.search(reg_zip, str(data))
    if zip_match:
        zipcode = str(zip_match.group(0).strip().replace(' ', '').lower())
    else:
        zipcode = str(str(data).split()[-1])
    return zipcode

In [ ]:
# Foncton de parsing


def find_num(data, casepostal, street_num_col):
    # Si la colonne 'rl0201ix'/'num_civ' n'est pas vide, num. civ. = 'rl0201ix'/'num_civ'
    if street_num_col != "":
        return street_num_col
    else:
        num = ""
        # If CP, there's no street number
        if casepostal is None or casepostal == '':
            for i in str(data.replace(",", " ")).split(" "):
                ## Check for the first string that contains either both digits and letters (ex: 34-A) or just digits
                if bool(re.match(reg_num, i)) is True or bool(re.match(reg_digits, i)) is True:
                    x = i
                    break
            ## If the string contains both digits and letters, the address is the digit part of the strign (ex : 34-A would be 34)
            if bool(re.match(reg_num, x)) is True:
                num = max(re.findall(reg_digits, x), key=len)
            else:
                ## If the string is just plain digits (ex : 1234), this is the address
                if x.isnumeric():
                    num = x
                else:
                    for i in str(data.replace(",", " ")).split(" "):
                        ## Check for the first string that contains either both digits and letters (ex: 34-A) or just digits
                        if bool(re.match(reg_num, i)) is True or bool(re.match(reg_digits, i)) is True:
                            x = i
                            break
                    if bool(re.match(reg_digits_hyphen, x)):
                        num = x.split('-')[1]
        return str(num).strip()


def find_case_postal(data, case_postal_col):
    # Si la colonne 'rl0201tx'/'cp' n'est pas vide, cp = 'rl0201tx'/'cp'
    if case_postal_col != "":
        return case_postal_col
    else:
        cp = None
        # Search if the address starts with CP or PO BOX
        if re.search(reg_cp, str(data)) is not None:
            for i in data.split():
                # The first digit is the CP
                if i.isdigit():
                    cp = i
                    return cp
                    break
            if cp is None:
                try:
                    cp = re.search(reg_cp, str(data)).group()
                    cp = cp.replace('cp', '').replace('c.p.', '').replace('po box', '').replace('case postale', '').strip()  # clean cp
                    return cp
                except:
                    pass


def find_suite(data, case_postal, street_num, card, street, road, zipcode, suite_col):
    # Si la colonne 'rl0201ox'/'num_app', n'est pas vide, suite = 'rl0201ox'/'num_app'
    if suite_col != "":
        return suite_col
    else:
        suite = None
        # If CP, there's no suite number, else...
        if case_postal is None or case_postal == '':
            if suite is None or suite == '':
                for i in str(data.replace(",", " ")).split(" "):
                    ## Check for the first string that contains either both digits and letters (ex: 34-A) or just digits
                    if bool(re.match(reg_num, i)) is True or bool(re.match(reg_digits, i)) is True:
                        x = i
                        break
                if bool(re.match(reg_digits_hyphen, x)):
                    suite = x.split('-')[0]
            if suite is None or suite == '':
                data = data.replace(street, '', 1).replace(zipcode, '').replace(street_num, '', 1).replace(road, '', 1)
                if card is not None:
                    data = data.replace(card, '')
                # Search for a combination of keywords followed by a combination of letters and/or digits separated or not by a dash (ex : A-30, 90E, 1234, A)
                suite = re.search(reg_suite, data.replace('.', ''))
                if suite is not None:
                    suite = suite.group().strip().strip('-')
                else:
                    if suite is None:
                        suite = re.search(reg_enieme, data.replace('.', ''))
                        if suite is not None:
                            suite = suite.group().strip()
                    if suite is None or suite == '':
                        for i in str(data.replace(",", " ")).split(" "):
                            # Check for the first string that contains both digits and letters (ex : 34-B, C23)
                            if bool(re.match(reg_num, i)) == True:
                                if (len(i) >= 2) and i != zipcode.lower():
                                    # The suite is the letter part of this string
                                    suite = i
                                    break
                    if suite is None or suite == '':
                        # Check for a stand-alone digit or letter in the address once the street name, street number, and zipcode have been removed
                        for i in str(data.replace(",", " ")).split(" "):
                            if i.isdigit():
                                suite = i
                            if suite is None and len(i) < 3 and i.isdigit() == False:
                                suite = i
                            if bool(re.match(r'^[A-Za-z]$', i)) is True:
                                suite = i
                suite = functools.reduce(lambda a, b: a.replace(b, ''), suite_stopwords, suite)
                if len(suite) == 3 and suite in zipcode:
                    return ""
                else:
                    return suite
            else:
                if len(str(suite)) == 3 and str(suite) in zipcode:
                    return ""
                else:
                    return str(suite)


def find_street(data, case_postal, num, street_col, roadtype):
    # --- nettoyage ---
    clean = str(data).lower()
    clean_comma = " ".join([re.sub(reg_st, r'\1st', x) for x in str(data).lower().split()])  ## replace saint by st
    clean = " ".join([re.sub(reg_ste, r'\1ste', x) for x in str(clean_comma).lower().split()])  ## replace sainte by ste
    clean = clean.replace(",", "").replace(".", "")
    words = clean.split()

    # Vérifie si la rue est une avenue (format regex : chiffre followed by avenue), si c'est le cas assigne le # comme nom de voie
    av = re.search(reg_avenue, clean)
    route = re.search(reg_route, clean)
    if av is not None:
        av = av.group()
        street = " ".join([word for word in str(av).replace(".", "").replace(",", "").replace(" -", "-").replace("- ", "-").split() if word.lower() not in road_type_fr])

    # Vérifie si la rue est une route (format regex : route followed by chiffre), si c'est le cas assigne le # comme nom de voie
    elif route is not None:
        route = route.group()
        street = " ".join([word for word in str(route).replace(".", "").replace(",", "").replace(" -", "-").replace("- ", "-").split() if word.lower() not in road_type_fr])

    # Si la colonne 'rl0201mx'/'rue', n'est pas vide ET que la colonne ne contient pas un type de voie, street = 'rl0201mx'/'rue'
    elif street_col != "" and any(words in street_col.split() for words in road_type_fr) == False:
        # --- remove stopwords (de, la, etc.) ---
        street_tokens = [w for w in street_col.split() if w not in stopwords]
        # --- remove directions ---
        street_tokens = [w for w in street_tokens if w not in dir]
        # --- remove suite stopwords (app, suite, no.) ---
        street_tokens = [w for w in street_tokens if w not in suite_stopwords]
        street_col = " ".join(street_tokens).strip()
        return street_col
    elif case_postal is not None:
        return ""
    else:
        if street_col != "":
            # Si la colonne 'rl0201mx'/'rue', n'est pas vide, mais contient un type de voie continuer...
            data = street_col
        else:
            data = data
        if len(words) == 1 and words[0] not in road_type_fr:
            street = words[0]
        elif roadtype != "":
            # Trouve la position du type de route dans la liste
            rt_idx = None
            for i, w in enumerate(words):
                if w in road_type_fr:
                    rt_idx = i
                    break
            if rt_idx is None:
                return ""

            rt = words[rt_idx].lower()
            # Si le type de voie est en anglais, le nom de la rue est le premier string qui précède le type de voie dans la liste
            if rt in road_type_en:
                street_tokens = [w for w in reversed(words[:rt_idx]) if not any(c.isdigit() for c in w)][:1]  # take first valid one

            # Sinon, le nom de la rue est le premier string qui suit le type de voie dans la liste
            else:
                street_tokens = words[rt_idx + 1:]

            # --- remove stopwords (de, la, etc.) ---
            street_tokens = [w for w in street_tokens if w not in stopwords]

            # --- remove suite stopwords (app, suite, no.) ---
            street_tokens = [w for w in street_tokens if w not in suite_stopwords]

            # --- remove directions ---
            street_tokens = [w for w in street_tokens if w not in dir]
            street = " ".join(street_tokens).strip()

            # --- remove digits and suite digits (ex: 825a or b234) ---
            street = ' '.join([token for token in street.split() if not re.fullmatch(r'\d+[a-zA-Z]?|[a-zA-Z]\d+', token)]).strip()
        else:
            # --- remove stopwords (de, la, etc.) ---
            street_tokens = [w for w in words if w not in stopwords]

            # --- remove directions ---
            street_tokens = [w for w in street_tokens if w not in dir]

            # --- remove suite stopwords (app, suite, no.) ---
            street_tokens = [w for w in street_tokens if w not in suite_stopwords]
            street = " ".join(street_tokens).strip()

            # --- remove digits and suite digits (ex: 825a or b234) ---
            street = ' '.join([token for token in street.split() if not re.fullmatch(r'\d+[a-zA-Z]?|[a-zA-Z]\d+', token)]).strip()
    return street


# Extrait le type de voie
def findRoadType(data,street_type_col):
    if street_type_col != "":
        return street_type_col.replace('.', '').lower().strip()
    array = str(data.replace(',', '')).split()
    print(array)
    streetType_en = ""
    # Vérifie si le type de voie est en anglais
    try:
        streetType = " ".join([word for word in array if word.lower() in road_type_en]).split()[0]
        return streetType
    except:
        try:
            array = str(data).split()
            streetType = " ".join([word for word in array if word.lower() in road_type_fr]).split()[0]
        except:
            streetType = ""

    return streetType.strip()


# Extrait l'orientation de la voie
def findStreetCard(data, card_col):
    if card_col != "":
        return card_col
    else:
        ## Same logic as the findRoadType() fonction
        substring = []
        for i in str(data).lower().replace(',', '').split():
            if len(i) == 1 and i == "o":
                i = i.replace("o", "ouest")
            if len(i) == 1 and i == "e":
                i = i.replace("e", "est")
            if len(i) == 1 and i == "n":
                i = i.replace("n", "nord")
            if len(i) == 1 and i == "s":
                i = i.replace("s", "sud")
            substring.append(i)
        clean = ' '.join(substring)
        array = clean.split()
        streetCard = " ".join([word for word in array if word.lower() in dir])
        streetCard.strip()
        if streetCard is not None:
            if streetCard == 'est':
                streetCard = streetCard.replace('est', 'e')
            if streetCard == 'ouest':
                streetCard = streetCard.replace('ouest', 'o')
            if streetCard == 'nord':
                streetCard = streetCard.replace('nord', 'n')
            if streetCard == 'sud':
                streetCard = streetCard.replace('sud', 's')
        if streetCard == '':
            streetCard = None
        return streetCard

In [ ]:
def parse_postal_address():
    data = pd.read_csv(path + csv, index_col=False, encoding='utf_8_sig')
    df = pd.DataFrame(data).fillna('')
    df = df.replace({"cpde_gen": replace_voie})
    print(f"Total rows: {len(df)}")

    # Filtrer pour copropriétés
    df = df[df['code_type'] == 3]
    print(f"Filtered rows: {len(df)}")

    total = len(df)
    parsed_data = []
    error = []

    for index, row in df.iterrows():
        try:
            address = row['adresse_post'].lower().replace('  ', ' ')
            street_type_col = row['cpde_gen'].lower().replace('  ', ' ')
            zipcode_col = str(row["code_post"]).lower().strip()
            case_postal_col = str(row["cp"]).lower().strip()
            street_num_col = str(row["num_civ"]).replace(".0", "").strip().lower()
            card_col = str(row["orient"]).lower().strip()
            street_col = str(row["rue"]).lower().strip()
            suite_col = str(row["num_app"]).lower().strip()

            clean_address = re.sub(r'(?<=[,])(?=[^\s])', r' ', address).replace('. -', '-').replace('.-', '').replace('- ', '-').replace('.', '')
            clean_address = unidecode.unidecode(clean_address)
            clean_address = ' '.join(clean_address.split())

            zipcode = clean_zip(zipcode_col)
            road = findRoadType(clean_address, street_type_col)
            case_postal = find_case_postal(clean_address, case_postal_col)
            street_num = find_num(clean_address, case_postal, street_num_col)
            card = findStreetCard(clean_address, card_col)
            street = find_street(clean_address, case_postal, street_num, street_col, road)
            suite = find_suite(clean_address, case_postal, street_num, card, street, road, zipcode, suite_col)

            parsed_data.append(row.tolist() + [street_num, road, street, card, suite, zipcode, case_postal])

        except Exception as e:
            error.append(row.tolist() + [str(e)])

    # Export parsed CSV
    new_data = pd.DataFrame(parsed_data)
    new_data.to_csv(path + 'address_proprio_parsed-v1.csv', index=False, encoding='utf_8_sig')
    print("Parsed data saved!")

    # Export errors
    error_data = pd.DataFrame(error)
    error_data.to_csv(path + 'address_parsed_error.csv', index=False, encoding='utf_8_sig')
    print("Errors saved!")

parse_postal_address()